In [1]:
!pip install pdfplumber
!git clone https://github.com/CyberMystic-Jude/File_MetaData_Extractor
!pip install yt_dlp
!pip install python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 32.8 MB/s eta 0:00:00
Cloning into 'File_MetaData_Extractor'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 36 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 9.77 KiB | 4.88 MiB/s, done.
Resolving deltas: 100% (10/10), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.4/182.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.6 MB/s eta 0:00:00


In [2]:
import pdfplumber
import os
from pathlib import Path
import pandas as pd
import numpy as np
import yt_dlp
import datetime
from docx import Document
import re
from collections import Counter
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
#tesseract


In [18]:
#Read PDF
def read_page(pdfName, output_folder):
  size = Path(pdfName).stat().st_size
  FileSize  = size / (1024) , 'KB'

  numWords = 0
  result = {
  "filename":Path(pdfName).name,
  "Number of Pages":"",
  "Source Type":Path(pdfName).parent.name,
  "File type":Path(pdfName).suffix.replace(".",""),
  "File size":FileSize,
  "Word count": "",
  "text" : "",
  #"tables" : [],
  "hasTables":""
  }


  with pdfplumber.open(pdfName) as pdf:
    numpages = len(pdf.pages)
    Filename = Path(pdfName).stem
    FileType = Path(pdfName).suffix.replace(".","")
    result["Number of Pages"] = numpages
    print("Filename: ", Filename)
    print("Number of Pages: ",numpages)
    print("File type:", FileType)
    print("File size:", FileSize)



    for i in range(numpages):
      page = pdf.pages[i]

      text = page.extract_text()
      word_count = len(text.split())
      numWords +=word_count
      result["text"] = text

      tables = page.extract_tables()
      if tables:
        result["hasTables"] = True
      else:
        result["hasTables"]= False
  print("Number of words:", numWords)

  result["Word count"] = numWords


  #if output_folder:
    #output_path = Path(output_folder) / (Path(pdfName).stem + ".csv")
  return result



In [19]:
def read_all_documents(parent_folder, sourcetype,output_folder):

  parent_path = Path(parent_folder)

  all_results = []

  for pdf_file in parent_path.glob(f"{sourcetype}/*.pdf"):
    print("\n=============================")
    print("Processing:", pdf_file.name)

    result = read_page(str(pdf_file), output_folder)

    #result["source_type"] = sourcetype
    result["document_id"] = pdf_file.stem

    all_results.append(result)



  return all_results



In [20]:
#Read video metadata and transcript https://hrekov.com/blog/youtube-metadata-python-yt-dlp

def get_youtube_metadata(url):
  ydl_opts = {
      'quiet': True,
      'skip_download': True,
      'no_warnings': True
      #'cookiesfrombrowser':('chrome',)
  }



  try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
      info = ydl.extract_info(url, download=False)

      metadata = {
          "Title": info.get('title'),
          "Views":info.get('view_count'),
          "Description":info.get('description'),
          "Author": info.get('uploader'),
          "text":"",
          "Duration": str(datetime.timedelta(seconds=info.get('duration',0))),
          "Upload Date": datetime.datetime.strptime(info.get('upload_date'), "%Y%m%d").strftime("%Y/%m/%d") if info.get('upload_date') else None
      }

      return metadata

  except Exception as e:
    print(f"Error fetching metadata: {e}")
    return None


In [21]:
#Read all video metadata

def get_video_metadata(parent_folder):

  video_folder = Path(parent_folder)
  video_doc_path = video_folder/ "YoutubeLinks.txt"
  #doc = Document(video_doc_path)
  #urls = [p.text for p in video_doc_path.paragraphs if p.text.strip()]
  urls = video_doc_path.read_text().splitlines()

  results = []
  #word_count = 0

  for url in urls:
    metadata = get_youtube_metadata(url)

    if metadata:
      metadata["URL"] = url
      title = metadata.get("Title")

      matches = list(video_folder.glob(f"Videos/{title}.txt"))

      if matches:
        transcript_path = matches[0]
        text = transcript_path.read_text(encoding="utf-8")
        metadata["text"] = text
        word_count = len(text.split())
      else:
        word_count = None

      metadata["Word Count"] = word_count

      results.append(metadata)

  return results

In [7]:
#def normalize_text(texts, word_mode=False):
  #words = line.split(' ')
 # words = line.lower()
  #words = re.sub(r'http\S+', '', words)
  #words = re.sub(r'<.*?>','',words)
  #words = re.sub(r'\([^)]*\)','',words)
  #words = re.sub(r'[.?!,;:"\']','',words)
  #words = re.sub(r'\s*\n\s*', '', words)
  #words = re.sub(r'[^a-z0-9\sáéíóúèêâăçèñšŧœ]', '', words)
  #line = ','.join(words)
  #line = re.sub(r'\s+', ' ', words).strip()
  #return line

In [8]:
#def cleaned_results(results):
#  for r in results:
#    text = r.get("text")
#    if text:
#      cleaned = normalize_text(text)
#      r["cleaned_text"] = cleaned
#  return results

In [9]:
nlp = spacy.load("en_core_web_sm")
stop_words = nlp.Defaults.stop_words
custom_words = {"video","policy","thank","government","committee","policies"}

stop_words = stop_words.union(custom_words)

In [22]:
def clean_text(text):
  if not text:
    return ""

  text = text.lower()
  text = re.sub(r"http\S+","",text)
  text = re.sub(r"\d+","", text)
  text = re.sub(r"[^\w\s]","", text)

  words = text.split()

  words = [w for w in words if w not in stop_words]

  return " ".join(words)

In [23]:
#frequent words and how often they appear
def frequent_words(results, top_n = 20):
  words = " ".join(results.dropna()).split()
  return Counter(words).most_common(top_n)
  #all_words= []

  #for r in results:
  #  text = r.get("cleaned_text")

   # if text:
   #   words = text.split()
   #   all_words.extend(words)

  #counts = Counter(all_words)

  #return counts.most_common(top_n)

In [16]:
def main():


  fileLocation = '/content/drive/MyDrive/MIT 808 Byte Squad/Data/'

  policies = "Policies"
  articles = "Articles"
  policies_metadata = read_all_documents(fileLocation, policies,fileLocation)
  articles_metadata = read_all_documents(fileLocation, articles,fileLocation)
  df_policies = pd.DataFrame(policies_metadata)
  df_articles = pd.DataFrame(articles_metadata)


  #top_words_policies = frequent_words(results_policies,20)


  videos = get_video_metadata(fileLocation)
  df_Videos = pd.DataFrame(videos)
  #for word, count in top_words_policies:
  #  print(word, count)
  results_policies = df_policies["text"].apply(clean_text)
  results_videos = df_Videos["text"].apply(clean_text)
  results_articles = df_articles["text"].apply(clean_text)

  print(results_policies.head())
  print("\n=============================")
  print("Policies")
  documents = results_policies.dropna().astype(str).tolist()

  vectorizer = TfidfVectorizer(stop_words="english")
  tfidf_matrix = vectorizer.fit_transform(documents)

  feature_names = vectorizer.get_feature_names_out()

  mean_scores = tfidf_matrix.mean(axis=0).A1

  top_indices = np.argsort(mean_scores)[::1][:20]

  for i in top_indices:
    print(feature_names[i], mean_scores[i])

  print("\n=============================")
  print("Videos")
  print(results_videos.head())
  vid_documents = results_policies.dropna().astype(str).tolist()

  vectorizer2 = TfidfVectorizer(stop_words="english")
  tfidf_matrix2 = vectorizer2.fit_transform(vid_documents)

  feature_names2 = vectorizer2.get_feature_names_out()

  mean_scores2 = tfidf_matrix2.mean(axis=0).A1

  top_indices2 = np.argsort(mean_scores2)[::1][:20]

  for i in top_indices2:
    print(feature_names2[i], mean_scores2[i])

  print("\n=============================")
  print("Articles")
  print(results_articles.head())
  art_documents = results_policies.dropna().astype(str).tolist()

  vectorizer3 = TfidfVectorizer(stop_words="english")
  tfidf_matrix3 = vectorizer3.fit_transform(art_documents)

  feature_names3 = vectorizer3.get_feature_names_out()

  mean_scores3 = tfidf_matrix3.mean(axis=0).A1

  top_indices3 = np.argsort(mean_scores3)[::1][:20]

  for i in top_indices3:
    print(feature_names3[i], mean_scores3[i])
  #print(head(df_Videos))

  df_all = pd.concat([results_videos,results_policies, results_articles], ignore_index=True)
  print(df_all["source"].value_counts())
  #results_policies = clean_text(policies_metadata["text"])


  print(df_policies.head())


In [24]:
if __name__ == "__main__":
  try:
    main()
  except StopIteration as e:
    print(f"Error: A StopIteration occurred. This likely means a required file (e.g., a .docx file for video URLs) was not found. Please ensure your data directory is correctly structured and contains the necessary files. Details: {e}")
  except Exception as e:
    print(f"An unexpected error occurred: {e}")


Processing: Protocol_on_Forestry_2002.pdf
Filename:  Protocol_on_Forestry_2002
Number of Pages:  25
File type: pdf
File size: (976.541015625, 'KB')
Number of words: 0

Processing: SADC_Forestry_Strategy_2010-2020-English.pdf
Filename:  SADC_Forestry_Strategy_2010-2020-English
Number of Pages:  48
File type: pdf
File size: (1242.9736328125, 'KB')
Number of words: 17054

Processing: SADC_Forestry_Strategy_2010-2020-French.pdf
Filename:  SADC_Forestry_Strategy_2010-2020-French
Number of Pages:  64
File type: pdf
File size: (1232.2607421875, 'KB')
Number of words: 19875

Processing: SADC_Forestry_Strategy_2010-2020-Portuguese.pdf
Filename:  SADC_Forestry_Strategy_2010-2020-Portuguese
Number of Pages:  59
File type: pdf
File size: (1324.0244140625, 'KB')
Number of words: 19098

Processing: SADC_Forest_Law_Enforcement_Governance_and_Trade_Program-English.pdf
Filename:  SADC_Forest_Law_Enforcement_Governance_and_Trade_Program-English
Number of Pages:  41
File type: pdf
File size: (375.935546

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
  fileLocation = '/content/drive/MyDrive/MIT 808 Byte Squad/Data/'

  policies = "Policies"
  articles = "Articles"
  policies_metadata = read_all_documents(fileLocation, policies,fileLocation)
  articles_metadata = read_all_documents(fileLocation, articles,fileLocation)
  df_policies = pd.DataFrame(policies_metadata)
  df_articles = pd.DataFrame(articles_metadata)


  #top_words_policies = frequent_words(results_policies,20)


  videos = get_video_metadata(fileLocation)
  df_Videos = pd.DataFrame(videos)
  #for word, count in top_words_policies:
  #  print(word, count)
  results_policies = df_policies["text"].apply(clean_text)
  results_videos = df_Videos["text"].apply(clean_text)
  results_articles = df_articles["text"].apply(clean_text)

  print(results_policies.head())
  print("\n=============================")
  print("Policies")
  documents = results_policies.dropna().astype(str).tolist()

  vectorizer = TfidfVectorizer(stop_words="english")
  tfidf_matrix = vectorizer.fit_transform(documents)

  feature_names = vectorizer.get_feature_names_out()

  mean_scores = tfidf_matrix.mean(axis=0).A1

  top_indices = np.argsort(mean_scores)[::1][:20]

  for i in top_indices:
    print(feature_names[i], mean_scores[i])

  print("\n=============================")
  print("Videos")
  print(results_videos.head())
  vid_documents = results_policies.dropna().astype(str).tolist()

  vectorizer2 = TfidfVectorizer(stop_words="english")
  tfidf_matrix2 = vectorizer2.fit_transform(vid_documents)

  feature_names2 = vectorizer2.get_feature_names_out()

  mean_scores2 = tfidf_matrix2.mean(axis=0).A1

  top_indices2 = np.argsort(mean_scores2)[::1][:20]

  for i in top_indices2:
    print(feature_names2[i], mean_scores2[i])

  print("\n=============================")
  print("Articles")
  print(results_articles.head())
  art_documents = results_policies.dropna().astype(str).tolist()

  vectorizer3 = TfidfVectorizer(stop_words="english")
  tfidf_matrix3 = vectorizer3.fit_transform(art_documents)

  feature_names3 = vectorizer3.get_feature_names_out()

  mean_scores3 = tfidf_matrix3.mean(axis=0).A1

  top_indices3 = np.argsort(mean_scores3)[::1][:20]

  for i in top_indices3:
    print(feature_names3[i], mean_scores3[i])
  #print(head(df_Videos))

  df_all = pd.concat([results_videos,results_policies, results_articles], ignore_index=True)
  print(df_all["source"].value_counts())


Processing: Protocol_on_Forestry_2002.pdf
Filename:  Protocol_on_Forestry_2002
Number of Pages:  25
File type: pdf
File size: (976.541015625, 'KB')
Number of words: 0

Processing: SADC_Forestry_Strategy_2010-2020-English.pdf
Filename:  SADC_Forestry_Strategy_2010-2020-English
Number of Pages:  48
File type: pdf
File size: (1242.9736328125, 'KB')
Number of words: 17054

Processing: SADC_Forestry_Strategy_2010-2020-French.pdf
Filename:  SADC_Forestry_Strategy_2010-2020-French
Number of Pages:  64
File type: pdf
File size: (1232.2607421875, 'KB')
Number of words: 19875

Processing: SADC_Forestry_Strategy_2010-2020-Portuguese.pdf
Filename:  SADC_Forestry_Strategy_2010-2020-Portuguese
Number of Pages:  59
File type: pdf
File size: (1324.0244140625, 'KB')
Number of words: 19098

Processing: SADC_Forest_Law_Enforcement_Governance_and_Trade_Program-English.pdf
Filename:  SADC_Forest_Law_Enforcement_Governance_and_Trade_Program-English
Number of Pages:  41
File type: pdf
File size: (375.935546

KeyError: 'source'

In [32]:
summary_stats = pd.DataFrame({
    "Metric": [
        "Average Policy Pages",
        "Average Policy Words",
        "Average Article Words",
        "Average Video Duration"
    ],
    "Value": [
        df_policies["Number of words"].mean(),
        df_policies["numwords"].mean(),
        df_articles["numwords"].mean(),
        df_Videos["duration"].mean()
    ]
})

print(summary_stats)

KeyError: 'Number of words'